# 10 — Cassandra: Modelagem e CQL

**Objetivos:** modelar tabelas orientadas às consultas e usar Cassandra real como prática opcional

**Como estudar:** leia a explicação, execute os exemplos, interprete os resultados, refaça os exercícios e consulte o resumo. Exemplos sintéticos são identificados; serviços externos são opcionais e precisam estar disponíveis para laboratórios de infraestrutura.

### Materiais complementares

- [Guia Técnico](./reference_materials/10_cassandra_modelagem_cql.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_cassandra_modelagem_cql.pdf)


In [1]:
from pathlib import Path
import pandas as pd
BASE = Path.cwd()
DATA = BASE / 'datasets'
OUT = BASE / 'outputs'
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
assert (DATA / 'leituras_sensores.csv').exists(), 'Abra o JupyterLab na raiz da disciplina (onde está datasets/).'


## Cassandra e padrões de acesso

Cassandra é um banco wide-column distribuído: modelamos a partir das perguntas. A chave de partição define agrupamento; clustering columns ordenam os dados dentro da partição. Joins e consultas ad hoc não seguem o mesmo desenho do SQL relacional.

In [2]:
print((Path('sql')/'04_cassandra.cql').read_text(encoding='utf8'))

-- Modelo ilustrativo para consultas definidas. Execute no cqlsh após subir Cassandra.
CREATE KEYSPACE IF NOT EXISTS transporte WITH replication = {'class':'SimpleStrategy','replication_factor':1};
USE transporte;
CREATE TABLE IF NOT EXISTS embarques_por_linha_dia (
  linha text, dia date, horario timestamp, id_evento uuid, ponto text, passageiros int,
  PRIMARY KEY ((linha,dia),horario,id_evento)
) WITH CLUSTERING ORDER BY (horario DESC,id_evento ASC);
CREATE TABLE IF NOT EXISTS embarques_por_ponto_dia (
  ponto text, dia date, horario timestamp, id_evento uuid, linha text, passageiros int,
  PRIMARY KEY ((ponto,dia),horario,id_evento)
) WITH CLUSTERING ORDER BY (horario DESC,id_evento ASC);
-- Repetir eventos em duas tabelas é intencional na modelagem orientada à consulta.



## Estudo de caso — transporte

Exercício inspirado no problema de rotas e embarque/desembarque. **C1:** listar eventos por linha e dia, do mais recente. **C2:** listar eventos por ponto e dia. Criamos duas tabelas orientadas às consultas, duplicando dados intencionalmente.

In [3]:
c1='SELECT * FROM transporte.embarques_por_linha_dia WHERE linha=? AND dia=?'; c2='SELECT * FROM transporte.embarques_por_ponto_dia WHERE ponto=? AND dia=?';print('C1:',c1,'\nC2:',c2)

C1: SELECT * FROM transporte.embarques_por_linha_dia WHERE linha=? AND dia=? 
C2: SELECT * FROM transporte.embarques_por_ponto_dia WHERE ponto=? AND dia=?


## Laboratório real opcional

Use `docker compose -f infra/compose.yaml --profile cassandra up -d`, aguarde o healthcheck, execute o script CQL via `cqlsh`. A consulta Python a seguir é habilitada apenas mediante confirmação explícita.

In [4]:
# A execução local não ativa conexões automaticamente.
# Para laboratórios opcionais, configure RUN_LIVE_DB=1 antes de iniciar o kernel,
# depois de verificar que a instância é EXCLUSIVAMENTE de estudos.
import os
print('Laboratório externo:', 'habilitado' if os.getenv('RUN_LIVE_DB') == '1' else 'desabilitado (padrão seguro)')

Laboratório externo: desabilitado (padrão seguro)


In [5]:
import os
if os.getenv('RUN_LIVE_DB')=='1':
 from cassandra.cluster import Cluster
 from cassandra.query import SimpleStatement
 cluster=Cluster(['127.0.0.1'],port=9042,connect_timeout=8)
 session=cluster.connect()
 print('Cluster:',session.execute('SELECT release_version FROM system.local').one())
 session.execute("CREATE KEYSPACE IF NOT EXISTS transporte WITH replication = {'class':'SimpleStrategy','replication_factor':1}")
 cluster.shutdown()
else:
 print('Cassandra não executado: é necessário contêiner real e RUN_LIVE_DB=1.')

Cassandra não executado: é necessário contêiner real e RUN_LIVE_DB=1.


## Exercício resolvido — particionamento

**Problema:** uma consulta por linha e dia deve evitar varrer toda a base. **Solução:** `(linha,dia)` é a partition key, `horario` a clustering key. **Limitação:** linhas com volume extremamente desigual exigem avaliar partições grandes e buckets.

In [6]:
print('Chave de partição: (linha, dia); ordenação dentro da partição: horario DESC')

Chave de partição: (linha, dia); ordenação dentro da partição: horario DESC


## Pontos de atenção

SimpleStrategy e fator de replicação 1 são exclusivos do laboratório local, NÃO recomendações para produção. Sem múltiplos nós, não afirmaremos observar tolerância a falhas distribuída.

## Consulta Rápida e critérios de revisão

- **Antes de executar:** confirme o diretório de trabalho e as dependências do notebook.
- **Antes de concluir:** separe fatos observados, hipóteses, limitações e resultados sintéticos.
- **Antes de publicar:** não inclua credenciais, dados pessoais, volumes de bancos ou PDFs privados.
- **Exercício adicional:** altere uma hipótese do exemplo e explique o que mudou na saída, com evidência do código.

---
### Aprofundamento v2 — Modelagem orientada à consulta

No Cassandra, a **chave de partição** define o agrupamento lógico; as *clustering columns* ordenam linhas dentro dessa partição. Uma chave popular demais cria partições quentes ou excessivamente grandes. Dados podem ser duplicados deliberadamente em diferentes tabelas para responder a consultas distintas.

**Interpretação:** compare o resultado com a teoria anterior; este exemplo local não substitui um laboratório de serviço externo.

In [7]:
eventos=[('L1','2026-11-09',8,'P1'),('L1','2026-11-09',9,'P2'),('L2','2026-11-09',10,'P1')]
por_linha={}
for linha,dia,hora,ponto in eventos:
    por_linha.setdefault((linha,dia),[]).append((hora,ponto))
assert len(por_linha[('L1','2026-11-09')])==2
print('Modelo lógico por chave (linha,dia):',por_linha)

Modelo lógico por chave (linha,dia): {('L1', '2026-11-09'): [(8, 'P1'), (9, 'P2')], ('L2', '2026-11-09'): [(10, 'P1')]}


**Exercício resolvido — revisão:** identifique a principal hipótese do exemplo acima, explique por que ela importa e cite uma condição na qual a técnica falharia. **Resposta orientadora:** registre a pré-condição (tipo de dado, chave, restrição, volume ou topologia), teste-a explicitamente e explique os limites dos resultados observados.

### Resultado reproduzível em `outputs/`

A célula abaixo exporta uma evidência local do estudo. Não conecta a servidores externos.

In [8]:
pd.DataFrame([{'linha':linha,'dia':dia,'registros':len(vals)} for (linha,dia),vals in sorted(por_linha.items())]).to_csv(OUT / '10_particoes_logicas_exemplo.csv',index=False)
print('Exportado: outputs/10_particoes_logicas_exemplo.csv')

Exportado: outputs/10_particoes_logicas_exemplo.csv
